### Agent 的高级用法

#### 1. 设置 agent 的名称

使用字段 name 来进行设置
举例: 

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
import os
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv(override=True)
ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_API_BASE_URL = os.getenv("ZHIPUAI_API_BASE_URL")

# 获取大模型
model = init_chat_model(
    model='glm-5.3',
    model_provider="openai",
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_API_BASE_URL
)

agent = create_agent(
    # 调用的是 init_chat_model
    model= model,
    name="agent01"
)

response = agent.invoke({"messages": [
    {
        "role": "system", 
        "content": "你是一个精通数学的老师, 擅长以通俗易懂的方式讲解数学问题"
    },
    {
        "role": "user", 
        "content": "100 + 20 * 3 = ?"
    }
]})
rprint(type(response))
rprint(response)

#### 2. 系统提示词的设置

使用 system_prompt 参数进行设置, 可以是 str, 也可以是 SystemMessage

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
import os
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_tavily import TavilySearch
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv(override=True)
ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_API_BASE_URL = os.getenv("ZHIPUAI_API_BASE_URL")

# 获取大模型
model = init_chat_model(
    model='glm-5.3',
    model_provider="openai",
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_API_BASE_URL
)

# 内置工具
web_search = TavilySearch(
    max_results=2,
    tavily_api_key=os.getenv("TAVILY_API_KEY")
)

agent = create_agent(
    # 调用的是 init_chat_model
    model= model,
    tools = [web_search],
    system_prompt="你是一名多才多艺的智能助手, 可以调用工具帮助用户解决问题"
)

response = agent.invoke({"messages": [
    {
        "role": "user", 
        "content": "请帮我查询 2024 年诺贝尔物理学奖得主是谁?"
    }
]})
rprint(response)

### 3. 结构化输出的 4 种策略

#### 3.1 ProviderStrategy

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
import os
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_tavily import TavilySearch
from dotenv import load_dotenv
from rich import print as rprint
from pydantic import BaseModel, Field

load_dotenv(override=True)
ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_API_BASE_URL = os.getenv("ZHIPUAI_API_BASE_URL")

# 1. 模型初始化
model = init_chat_model(
    model='glm-5.3',
    model_provider="openai",
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_API_BASE_URL
)

# 2. 使用 Pydantic 结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户的名称")
    email: str = Field(description="用户的邮箱")
    phone: str = Field(description="用户的电话")

agent = create_agent(
    model=model,
    response_format=ProviderStrategy(ContractInfo)
)

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "从一下信息中提取用户信息, 小明的邮箱是 shkstart@atguigu.com, 电话是 13012341234"
        }
    ]
})

rprint(response)

#### 3.2 ToolStrategy

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
import os
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_tavily import TavilySearch
from dotenv import load_dotenv
from rich import print as rprint
from pydantic import BaseModel, Field

load_dotenv(override=True)
ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_API_BASE_URL = os.getenv("ZHIPUAI_API_BASE_URL")

# 1. 模型初始化
model = init_chat_model(
    model='glm-5.3',
    model_provider="openai",
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_API_BASE_URL
)

# 2. 使用 Pydantic 结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户的名称")
    email: str = Field(description="用户的邮箱")
    phone: str = Field(description="用户的电话")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(ContractInfo)
)

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "从一下信息中提取用户信息, 小明的邮箱是 shkstart@atguigu.com, 电话是 13012341234"
        }
    ]
})

rprint(response)

### 3.3 type/AutoStrategy

In [2]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
import os
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_tavily import TavilySearch
from dotenv import load_dotenv
from rich import print as rprint
from pydantic import BaseModel, Field

load_dotenv(override=True)
ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_API_BASE_URL = os.getenv("ZHIPUAI_API_BASE_URL")

# 1. 模型初始化
model = init_chat_model(
    model='glm-5.3',
    model_provider="openai",
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_API_BASE_URL
)

# 2. 使用 Pydantic 结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户的名称")
    email: str = Field(description="用户的邮箱")
    phone: str = Field(description="用户的电话")

agent = create_agent(
    model=model,
    response_format=AutoStrategy(ContractInfo)
)

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "从一下信息中提取用户信息, 小明的邮箱是 shkstart@atguigu.com, 电话是 13012341234"
        }
    ]
})

rprint(response)

{
    'messages': [
        HumanMessage(
            content='从一下信息中提取用户信息, 小明的邮箱是 shkstart@atguigu.com, 电话是 13012341234',
            additional_kwargs={},
            response_metadata={},
            id='c763ae52-141f-45f9-8396-86a2c89a93f3'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 97,
                    'prompt_tokens': 224,
                    'total_tokens': 321,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 62,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.3',
                'system_fingerprint': None,
                'id': '20261113143519f2f8566ca07d41a4',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1d39e-2625-7ca0-9415-f89c5402eef7-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'email': 'shkstart@atguigu.com', 'name': '小明', 'phone': '13012341234'},
                    'id': 'call_-7080017914010529029',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 224,
                'output_tokens': 97,
                'total_tokens': 321,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 62}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='shkstart@atguigu.com' phone='13012341234'",
            name='ContractInfo',
            id='cea920d7-f262-4deb-85b5-1153db0eb8d8',
            tool_call_id='call_-7080017914010529029'
        )
    ],
    'structured_response': ContractInfo(name='小明', email='shkstart@atguigu.com', phone='13012341234')
}